# Day 40 — Baseline LSTM Classifier on Dynamic Features

Train an LSTM classifier on API call sequences extracted from CAPEv2 reports.
Uses the tokenizer from Day 39 (`models/tokenizer.py`) and the architecture
validated in the Day 38 refresher.

Since Day 35 (full corpus) is pending, we demonstrate on synthetic data
generated from the two fixture reports. The architecture and training loop
are production-ready — swap in real data when available.

**Architecture:** Embedding(64) → LSTM(128, 2 layers) → Linear(2)

**Metrics:** accuracy, precision, recall, F1, AUC-ROC

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

from dynamic_sandbox.api_sequence import extract_flat_sequence
from models.tokenizer import APITokenizer

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Data & Build Tokenizer

In production, load from `data/processed/api_sequences_clean.json`.
Here we generate synthetic data from the fixture report patterns.

In [ ]:
fixtures_dir = os.path.join('..', 'tests', 'fixtures')

with open(os.path.join(fixtures_dir, 'cape_report_benign.json')) as f:
    benign_report = json.load(f)
with open(os.path.join(fixtures_dir, 'cape_report_suspicious.json')) as f:
    susp_report = json.load(f)

benign_apis = extract_flat_sequence(benign_report)
susp_apis = extract_flat_sequence(susp_report)

print(f'Benign API sequence length: {len(benign_apis)}')
print(f'Suspicious API sequence length: {len(susp_apis)}')

In [ ]:
def augment_sequence(base_seq, n_samples=100, min_frac=0.5, max_frac=1.0, noise_rate=0.1):
    """Generate synthetic variations of a base API sequence."""
    samples = []
    api_pool = list(set(base_seq))
    for _ in range(n_samples):
        frac = random.uniform(min_frac, max_frac)
        length = max(3, int(len(base_seq) * frac))
        seq = []
        for i in range(length):
            if i < len(base_seq) and random.random() > noise_rate:
                seq.append(base_seq[i % len(base_seq)])
            else:
                seq.append(random.choice(api_pool))
        samples.append(seq)
    return samples

N_PER_CLASS = 150
benign_samples = augment_sequence(benign_apis, N_PER_CLASS)
susp_samples = augment_sequence(susp_apis, N_PER_CLASS)

all_sequences = benign_samples + susp_samples
all_labels = [0] * N_PER_CLASS + [1] * N_PER_CLASS

tokenizer = APITokenizer(max_len=128, min_freq=1)
tokenizer.build_vocab(all_sequences)
info = tokenizer.summary()
print(f'Vocabulary size: {info["vocab_size"]}')
print(f'Max sequence length: {info["max_len"]}')
print(f'Total samples: {len(all_sequences)} ({N_PER_CLASS} per class)')

In [ ]:
indices = list(range(len(all_sequences)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

class APISeqDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer):
        self.encoded = [tokenizer.encode_padded(seq) for seq in sequences]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (torch.tensor(self.encoded[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

train_seqs = [all_sequences[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]
test_seqs = [all_sequences[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

train_ds = APISeqDataset(train_seqs, train_labels, tokenizer)
test_ds = APISeqDataset(test_seqs, test_labels, tokenizer)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

print(f'Train: {len(train_ds)}, Test: {len(test_ds)}')
print(f'Train class balance: {Counter(train_labels)}')
print(f'Test class balance: {Counter(test_labels)}')

## 2. LSTM Model

In [ ]:
class DynamicLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128,
                 num_layers=2, num_classes=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        emb = self.dropout(emb)
        lstm_out, (h_n, _) = self.lstm(emb)
        last_hidden = self.dropout(h_n[-1])
        logits = self.fc(last_hidden)
        return logits

model = DynamicLSTM(
    vocab_size=tokenizer.vocab_size,
    embed_dim=64,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3,
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## 3. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 30
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for seqs, labels in train_loader:
        seqs, labels = seqs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(seqs)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

    scheduler.step()
    train_loss = total_loss / total
    train_acc = correct / total
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for seqs, labels in test_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            logits = model(seqs)
            loss = criterion(logits, labels)
            val_loss += loss.item() * len(labels)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total += len(labels)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:>2}/{EPOCHS}  '
              f'train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  '
              f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}')

## 4. Evaluation

In [ ]:
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix)

model.eval()
all_preds = []
all_probs = []
all_true = []

with torch.no_grad():
    for seqs, labels in test_loader:
        seqs = seqs.to(device)
        logits = model(seqs)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_true.extend(labels.numpy())

print('=== Classification Report ===')
print(classification_report(all_true, all_preds,
                            target_names=['benign', 'malicious']))

try:
    auc = roc_auc_score(all_true, all_probs)
    print(f'AUC-ROC: {auc:.4f}')
except ValueError:
    print('AUC-ROC: could not compute (single class in test set)')

cm = confusion_matrix(all_true, all_preds)
print(f'\nConfusion Matrix:\n{cm}')

## 5. Training Curves

In [ ]:
print('Epoch | Train Loss | Train Acc | Val Loss | Val Acc')
print('-' * 55)
for i in range(len(history['train_loss'])):
    if (i + 1) % 5 == 0 or i == 0:
        print(f'{i+1:>5} | {history["train_loss"][i]:>10.4f} | '
              f'{history["train_acc"][i]:>9.3f} | '
              f'{history["val_loss"][i]:>8.4f} | '
              f'{history["val_acc"][i]:>7.3f}')

## Notes

- On synthetic data, the LSTM should converge quickly to near-perfect accuracy
  because the benign/malicious API vocabularies are largely disjoint.
- On real data (after Day 35), expect lower accuracy — real benign and malicious
  sequences share many common APIs (LdrLoadDll, NtCreateFile, etc.).
- The LSTM captures **temporal ordering** — injection chains like
  `NtAllocateVirtualMemory → NtWriteVirtualMemory → CreateRemoteThread`
  are strong signals that tabular features miss.
- Hyperparameters to tune on real data: embed_dim, hidden_dim, num_layers,
  dropout, learning rate, sequence length, batch size.